# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ismayilysfli/FlyRank-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

It is a ranking task because we are trying to identify highest-priority pages for review

In [1]:
import os
import subprocess
import pandas as pd

REPO_URL = "https://github.com/ismayilysfli/FlyRank-ml.git"
REPO_DIR = "/content/FlyRank-ml"

if not os.path.isdir(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
        check=True
    )

os.chdir(REPO_DIR)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset loaded:", df.shape)

Dataset loaded: (30000, 44)


## 2. Target or proxy

For the starter dataset, I would use whether a page is declining as a proxy for review priority. The proxy can be defined from trend_direction == "down". This is useful for the first version of the project, but it is not an ideal final target because it is derived from the current trend data rather than from a future observed outcome. A stronger target later would use past data to predict whether a page declines or recovers in a future time window.

In [2]:
df["decline_proxy"] = (df["trend_direction"] == "down").astype(int)

df[["content_id", "trend_direction", "decline_proxy"]].head()

,content_id,trend_direction,decline_proxy
0,content_304f48230142,down,1
1,content_a1fb4e703a9e,down,1
2,content_9aa793d4d895,down,1
3,content_331d6c4de07b,stable,0
4,content_d99b7a2d90ca,down,1


## 3. Success metric

I would use Precision@50 as the main success metric because the goal is to rank the most useful pages at the top of a limited review queue. Precision@50 measures how many of the top 50 recommended pages match the positive target or proxy. A good result should outperform the existing fixed-rule baseline on held-out data. In the starter pipeline, the baseline Precision@50 is 0.24, so a useful model should perform better than 0.24.

In [3]:
with open("outputs/model_report.md", "r") as f:
    report = f.read()

for line in report.splitlines():
    if line.startswith("| random_forest") or line.startswith("| baseline_rules"):
        parts = [x.strip() for x in line.strip("|").split("|")]

        model_name = parts[0]
        precision_at_50 = float(parts[3])

        print(f"{model_name} Precision@50: {precision_at_50:.3f}")

random_forest Precision@50: 0.740
baseline_rules Precision@50: 0.240


## 4. The unit of analysis, as a real dataframe

The unit of analysis is one content item/page. Each row in the starter dataset represents one pseudonymized page with its search, traffic, engagement, content, and trend-related measurements.

In [4]:
lane_df = df[
    [
        "content_id",
        "impressions_90d",
        "clicks_90d",
        "avg_position",
        "ctr",
        "word_count",
        "trend_direction",
        "decline_proxy"
    ]
].copy()

print(f"Rows: {len(lane_df):,}")
print("One row = one content item/page")

lane_df.head()

Rows: 30,000
One row = one content item/page


,content_id,impressions_90d,clicks_90d,avg_position,ctr,word_count,trend_direction,decline_proxy
0,content_304f48230142,3803,29,10.6,0.76,3221.0,down,1
1,content_a1fb4e703a9e,15320,7,20.3,0.05,2481.0,down,1
2,content_9aa793d4d895,12581,11,36.5,0.09,3515.0,down,1
3,content_331d6c4de07b,11751,58,6.2,0.49,NaN,stable,0
4,content_d99b7a2d90ca,19140,24,44.0,0.13,2803.0,down,1


## 5. Why ML beats a fixed rule here

A fixed rule is not enough because review priority depends on several signals such as impressions, position, CTR, freshness, engagement, and trend. The relationship between these signals is not simple, and one threshold cannot capture all useful combinations. For example, a page may have high impressions but low CTR, or good position but declining traffic. A learned model can combine these signals and rank pages more flexibly than a single hand-written rule.

In [5]:
print(
    "Correlation between search volume and impressions:",
    round(df["search_volume"].corr(df["impressions_90d"]), 3)
)

print("\nMedian word count by trend:")
print(
    df.groupby("trend_direction")["word_count"]
      .median()
      .loc[["down", "stable", "up"]]
)

Correlation between search volume and impressions: 0.001

Median word count by trend:
trend_direction
down      2909.0
stable    2912.5
up        2847.5
Name: word_count, dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.